# 2026-03 Basics of docling

- Example and fake data used for this study:
    - Microsoft Azure invoices: https://www.microsoft.com/en-us/download/details.aspx?id=38805
    - Kaggle receipts (2024-EN): https://www.kaggle.com/datasets/jenswalter/receipts

**Note**: Install docling with `uv pip install docling`. The first run will also download AI models (~1 GB) for layout analysis and table detection — this happens automatically and the models are cached for subsequent runs.

---

### Setup
Importing all necessary libs and getting a list with all the example data

In [1]:
# Install docling (run once)
# !uv pip install docling

In [2]:
# Python imports
import os, sys
from pathlib import Path

# Data handling imports
import pandas as pd
import numpy as np
from tqdm import tqdm

# --> Main imports: docling
from docling.document_converter import DocumentConverter
from docling.datamodel.base_models import ConversionStatus
from docling.chunking import HybridChunker

In [3]:
# Path to all the example files
path = Path('./data/')

# Get all files
files_list = [f'./data/{entry.name}' for entry in path.rglob('*') if entry.is_file()]

# Sort the list
files_list = sorted(files_list)

### Helper functions

In [4]:
def helper_doc_report(result):
    """
    Helper function to print a basic summary of a docling ConversionResult.
    """
    doc = result.document

    print("="*80)
    print(f"Source: {result.input.file}")
    print(f"Status: {result.status}")
    print("="*80)
    print(f"Texts:    {len(doc.texts)}")
    print(f"Tables:   {len(doc.tables)}")
    print(f"Pictures: {len(doc.pictures)}")
    print("")

In [5]:
def helper_conversion_report(result):
    """
    Helper function to print a markdown preview of a converted document.
    """
    md = result.document.export_to_markdown()
    lines = [l for l in md.splitlines() if l.strip()]
    words = md.split()

    print("="*80)
    print(f"Markdown export preview")
    print(f"Lines: {len(lines)} | Words: {len(words)} | Chars: {len(md)}")
    print("="*80)
    print(md[:500])
    print("")

### Basic testing
Just simple 'out of the box' examples and pipelines

In [6]:
# Initialize the DocumentConverter (downloads AI models on first run)
converter = DocumentConverter()

# Convert the cheesecakefactory receipt
result = converter.convert(files_list[1])

helper_doc_report(result)

Source: example-receipts-kaggle-cheesecakefactory.pdf
Status: ConversionStatus.SUCCESS
Texts:    36
Tables:   0
Pictures: 1



In [7]:
# Conversion status and document element counts
print(f"Status: {result.status}")
print(f"Successful: {result.status == ConversionStatus.SUCCESS}")

helper_conversion_report(result)

Status: ConversionStatus.SUCCESS
Successful: True
Markdown export preview
Lines: 36 | Words: 144 | Chars: 978
心

## ||上CI1t[5上CA|&lt;[ | ACIORY SEA｢｢LE

## O235 TABLE 203 #Party DALTON I\_ SVrCk: 5 17152 BAR DININ{:i 1 05/?0/24

1 HH Nac &amp; JaCks Dr

1 Louisi(1na Chicken Pasta

1 Nac &amp; %lacks Afi､ican Amb

1 Fresh Sti"dwberl y CC

5．50

25．95

9,00

12.95

Sub Total :

53.40

TaX :

5，53

05/20 18 : 40~I~O~TAI\_ :

58.93

GI-atuity Nut lI1clUded

Su99este(1 (j｢dtuity:

22% 1IP

12@96

20% IIP

11:79

18% 11P

1(1.61

Use your Phone's camera or visit https : //scar,QR,1o to scan the code below and



### Text Extraction
Accessing document content via `export_to_markdown()`, `doc.texts`, and `export_to_dict()`

In [8]:
# export_to_markdown() — the most convenient structured text output
doc = result.document
md = doc.export_to_markdown()

print("=== export_to_markdown() ===")
print(md[:800])

=== export_to_markdown() ===
心

## ||上CI1t[5上CA|&lt;[ | ACIORY SEA｢｢LE

## O235 TABLE 203 #Party DALTON I\_ SVrCk: 5 17152 BAR DININ{:i 1 05/?0/24

1 HH Nac &amp; JaCks Dr

1 Louisi(1na Chicken Pasta

1 Nac &amp; %lacks Afi､ican Amb

1 Fresh Sti"dwberl y CC

5．50

25．95

9,00

12.95

Sub Total :

53.40

TaX :

5，53

05/20 18 : 40~I~O~TAI\_ :

58.93

GI-atuity Nut lI1clUded

Su99este(1 (j｢dtuity:

22% 1IP

12@96

20% IIP

11:79

18% 11P

1(1.61

Use your Phone's camera or visit https : //scar,QR,1o to scan the code below and pay your check

<!-- image -->

We'ii lwb l() lledi dI)(川l yULII" vitjlt! l.nj IAj W .心心fal""-vey -心。Im Ililel、thiE&gt; c(I(le witllil-1 5 ,｣dｿ3： 3043-0(1052 -2405Z

I(\_llll lliぅli'｢ lll ''l'(jl', Sqt/SIIII IInt 2IJI(|

i

l

+.+.+ l..+ {､l.州糾､#:+:"�MH i".+**.↓神*＋,仲＊＊.f＊ｲ､1.4-*4.4

Rewd i､〔lsNelll


In [9]:
# .texts — iterate over all text elements with their label (heading, paragraph, etc.)
doc = result.document

print(f"=== Text elements: {len(doc.texts)} ===\n")
for i, text_item in enumerate(doc.texts[:12]):
    print(f"[{i:2d}] label={str(text_item.label):20s} | text={text_item.text[:60]!r}")

=== Text elements: 36 ===

[ 0] label=text                 | text='心'
[ 1] label=section_header       | text='||上CI1t[5上CA|<[ | ACIORY SEA｢｢LE'
[ 2] label=section_header       | text='O235 TABLE 203 #Party DALTON I_ SVrCk: 5 17152 BAR DININ{:i '
[ 3] label=text                 | text='1 HH Nac & JaCks Dr'
[ 4] label=text                 | text='1 Louisi(1na Chicken Pasta'
[ 5] label=text                 | text='1 Nac & %lacks Afi､ican Amb'
[ 6] label=text                 | text='1 Fresh Sti"dwberl y CC'
[ 7] label=text                 | text='5．50'
[ 8] label=text                 | text='25．95'
[ 9] label=text                 | text='9,00'
[10] label=text                 | text='12.95'
[11] label=text                 | text='Sub Total :'


In [10]:
# export_to_dict() — full structured representation as a Python dict / JSON
doc = result.document
doc_dict = doc.export_to_dict()

print("=== export_to_dict() — top-level keys ===")
print(list(doc_dict.keys()))
print()

# The 'body' contains the ordered list of content items
body = doc_dict.get('body', [])
print(f"Body items: {len(body)}\n")
for item in body[:4]:
    print(f"  {str(item)[:120]}")

=== export_to_dict() — top-level keys ===
['schema_name', 'version', 'name', 'origin', 'furniture', 'body', 'groups', 'texts', 'pictures', 'tables', 'key_value_items', 'form_items', 'pages']

Body items: 5



KeyError: slice(None, 4, None)

### Table Extraction
Accessing detected tables via `doc.tables`, exporting them as Markdown or pandas DataFrames

In [11]:
# Convert the invoice — more likely to contain structured tables
result_invoice = converter.convert(files_list[0])
doc_invoice = result_invoice.document

helper_doc_report(result_invoice)

# Quick overview: table count and markdown representation of each table
print(f"=== Tables found: {len(doc_invoice.tables)} ===\n")
for i, table in enumerate(doc_invoice.tables):
    md_table = table.export_to_markdown()
    print(f"Table {i} (markdown):\n{md_table[:300]}\n")

Usage of TableItem.export_to_markdown() without `doc` argument is deprecated.
Usage of TableItem.export_to_markdown() without `doc` argument is deprecated.


Source: example-invoice-microsoft-azure-PAYG.pdf
Status: ConversionStatus.SUCCESS
Texts:    73
Tables:   2
Pictures: 9

=== Tables found: 2 ===

Table 0 (markdown):
| Previous balance                                  |   773.39 |
|---------------------------------------------------|----------|
| Payments                                          |     0    |
| Outstanding balance (from previous billing cycle) |  -102.49 |
| Current Charges                       

Table 1 (markdown):
| Name            | Type          | Resource                          | Region   |   Consumed |   Included |   Billable |   Rate |   Value |
|-----------------|---------------|-----------------------------------|----------|------------|------------|------------|--------|---------|
| Networking      



In [12]:
# Extract each table as a pandas DataFrame
doc = result.document
print(f"=== Tables found: {len(doc.tables)} ===\n")

for i, table in enumerate(doc.tables):
    print(f"Table {i} | bbox: {table.prov[0].bbox if table.prov else 'n/a'}")

    # export_to_dataframe() returns a pd.DataFrame with the table cells
    df_table = table.export_to_dataframe()
    print(f"  Shape: {df_table.shape}")
    display(df_table)
    print()

=== Tables found: 0 ===



### Document Export
Persisting converted documents to disk in different formats (Markdown and JSON)

In [13]:
# Save the markdown export to a file
md_path = Path("./docling_export.md")
md_path.write_text(result.document.export_to_markdown())

print(f"Markdown saved to: {md_path}")
print(f"File size: {md_path.stat().st_size} bytes")

Markdown saved to: docling_export.md
File size: 1082 bytes


In [14]:
# Save the full document as JSON
import json

doc_dict = result.document.export_to_dict()
json_path = Path("./docling_export.json")
json_path.write_text(json.dumps(doc_dict, indent=2, ensure_ascii=False))

print(f"JSON saved to: {json_path}")
print(f"Top-level keys: {list(doc_dict.keys())}")

JSON saved to: docling_export.json
Top-level keys: ['schema_name', 'version', 'name', 'origin', 'furniture', 'body', 'groups', 'texts', 'pictures', 'tables', 'key_value_items', 'form_items', 'pages']


### Chunking
Splitting documents into semantically coherent chunks with `HybridChunker` — useful for RAG pipelines and LLM context windows

In [15]:
# HybridChunker splits the document into semantically coherent chunks
# useful for feeding into embedding models or LLM context windows
doc = result.document
chunker = HybridChunker()
chunks = list(chunker.chunk(doc))

print(f"=== HybridChunker: {len(chunks)} chunk(s) produced ===\n")
print(f"First chunk text preview:\n  {chunks[0].text[:200]!r}" if chunks else "No chunks produced")

=== HybridChunker: 4 chunk(s) produced ===

First chunk text preview:
  '心'


In [16]:
# Iterate over chunks — each chunk has .text and metadata about its origin
doc = result.document
chunker = HybridChunker()
chunks = list(chunker.chunk(doc))

print(f"=== Chunks: {len(chunks)} total ===\n")
for i, chunk in enumerate(chunks[:5]):
    print(f"Chunk {i}:")
    print(f"  Text ({len(chunk.text)} chars): {chunk.text[:120]!r}")
    print()

=== Chunks: 4 total ===

Chunk 0:
  Text (1 chars): '心'

Chunk 1:
  Text (365 chars): '1 HH Nac & JaCks Dr\n1 Louisi(1na Chicken Pasta\n1 Nac & %lacks Afi､ican Amb\n1 Fresh Sti"dwberl y CC\n5．50\n25．95\n9,00\n12.95'

Chunk 2:
  Text (376 chars): 'We\'ii lwb l() lledi dI)(川l yULII" vitjlt! l.nj IAj W .心心fal""-vey -心。Im Ililel、thiE> c(I(le witllil-1 5 ,｣dｿ3： 3043-0(10'

Chunk 3:
  Text (53 chars): "卜・+'ド!'.;ﾄ:*＊ﾎﾄ.#:＊*I.;*州＊州.ｷ.*************#l:1.1****"



### Batch Processing
Converting all documents and collecting structural metadata into a summary DataFrame

In [17]:
# Convert all PDFs and build a summary DataFrame
converter = DocumentConverter()
results_batch = []

for file_path in tqdm(files_list):
    try:
        res = converter.convert(file_path)
        doc = res.document
        results_batch.append({
            'file': Path(file_path).name,
            'status': str(res.status),
            'texts': len(doc.texts),
            'tables': len(doc.tables),
            'pictures': len(doc.pictures),
            'word_count': len(doc.export_to_markdown().split()),
        })
    except Exception as e:
        results_batch.append({'file': Path(file_path).name, 'error': str(e)})

df_batch = pd.DataFrame(results_batch)
display(df_batch)

100%|███████████████████████████████████████████████████████████████████████████████████████████████████| 8/8 [00:09<00:00,  1.14s/it]


,file,status,texts,tables,pictures,word_count
0,example-invoice-microsoft-azure-PAYG.pdf,ConversionStatus.SUCCESS,73,2,9,515
1,example-receipts-kaggle-cheesecakefactory.pdf,ConversionStatus.SUCCESS,36,0,1,144
2,example-receipts-kaggle-luckylouie.pdf,ConversionStatus.SUCCESS,171,0,1,340
3,example-receipts-kaggle-mcdonald.pdf,ConversionStatus.SUCCESS,19,1,0,210
4,example-receipts-kaggle-oldnavy.pdf,ConversionStatus.SUCCESS,19,1,0,489
5,example-receipts-kaggle-spaceneedle.pdf,ConversionStatus.SUCCESS,27,0,1,78
6,example-receipts-kaggle-wholefoods_1.pdf,ConversionStatus.SUCCESS,3,1,0,171
7,example-receipts-kaggle-wholefoods_2.pdf,ConversionStatus.SUCCESS,42,0,2,196
